In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

DATA_PATH = Path(
    "data/processed/business_model_2y_features.csv"
)

model_df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    low_memory=False
)

model_df["인허가일자"] = pd.to_datetime(
    model_df["인허가일자"],
    errors="coerce"
)

print(model_df.shape)
print(model_df["target_2y_closure"].value_counts())

(22873, 61)
target_2y_closure
0    21581
1     1292
Name: count, dtype: int64


In [3]:
import numpy as np
import pandas as pd

# 날짜 변환
model_df["인허가일자"] = pd.to_datetime(
    model_df["인허가일자"],
    errors="coerce"
)

model_df["폐업일자"] = pd.to_datetime(
    model_df["폐업일자"],
    errors="coerce"
)

# 개업 시점 변수 생성
model_df["open_year"] = model_df["인허가일자"].dt.year
model_df["open_month"] = model_df["인허가일자"].dt.month
model_df["open_quarter"] = model_df["인허가일자"].dt.quarter

# 주소 생성
model_df["analysis_address"] = (
    model_df["도로명주소"]
    .fillna(model_df["지번주소"])
    .astype("string")
    .str.strip()
)

# 동일 주소 과거 폐업 횟수 생성
model_df["same_address_history_count"] = 0

for address, group in model_df.groupby(
    "analysis_address",
    dropna=True
):
    close_dates = (
        group["폐업일자"]
        .dropna()
        .sort_values()
        .to_numpy(dtype="datetime64[ns]")
    )

    if len(close_dates) == 0:
        continue

    open_dates = (
        group["인허가일자"]
        .to_numpy(dtype="datetime64[ns]")
    )

    counts = np.searchsorted(
        close_dates,
        open_dates,
        side="left"
    )

    model_df.loc[
        group.index,
        "same_address_history_count"
    ] = counts

required_cols = [
    "open_year",
    "open_month",
    "open_quarter",
    "same_address_history_count"
]

print(model_df[required_cols].head())
print(model_df[required_cols].isna().sum())

   open_year  open_month  open_quarter  same_address_history_count
0       1955           6             2                           0
1       1981          11             4                           0
2       1982           2             1                           0
3       1982           5             2                           0
4       1982          10             4                           0
open_year                     0
open_month                    0
open_quarter                  0
same_address_history_count    0
dtype: int64


In [4]:
model_df.to_csv(
    "data/processed/business_model_2y_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("누락 컬럼 추가 후 다시 저장 완료")

누락 컬럼 추가 후 다시 저장 완료


In [5]:
categorical_cols = [
    "sido",
    "sigungu",
    "공사립구분명"
]

base_numeric_cols = [
    "open_year",
    "open_month",
    "open_quarter",
    "소재지면적",
    "사무실면적",
    "지도자수",
    "탈의실면적",
    "휴게실면적",
    "same_address_history_count"
]

dynamic_numeric_cols = [
    "prior_1y_local_open_count",
    "prior_1y_local_close_count",
    "prior_3y_local_open_count",
    "prior_3y_local_close_count",
    "prior_3y_close_to_open_ratio",
    "prior_3y_closure_pressure",
    "local_active_at_open",
    "local_net_change_1y",
    "log_local_active_at_open",
    "log_prior_1y_open_count",
    "log_prior_1y_close_count",
    "log_prior_3y_open_count",
    "log_prior_3y_close_count"
]

numeric_cols = base_numeric_cols + dynamic_numeric_cols
feature_cols = categorical_cols + numeric_cols

model_df = (
    model_df
    .sort_values("인허가일자")
    .reset_index(drop=True)
)

X = model_df[feature_cols].copy()
y = model_df["target_2y_closure"].astype(int)

for col in categorical_cols:
    X[col] = X[col].fillna("unknown").astype(str)

for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

In [6]:
n = len(model_df)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_valid = X.iloc[train_end:valid_end].copy()
y_valid = y.iloc[train_end:valid_end].copy()

X_test = X.iloc[valid_end:].copy()
y_test = y.iloc[valid_end:].copy()

# 숫자형 결측은 훈련 데이터 중앙값으로 채움
for col in numeric_cols:
    median_value = X_train[col].median()

    X_train[col] = X_train[col].fillna(median_value)
    X_valid[col] = X_valid[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

cat_feature_indices = [
    X.columns.get_loc(col)
    for col in categorical_cols
]

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Valid:", X_valid.shape, y_valid.value_counts().to_dict())
print("Test :", X_test.shape, y_test.value_counts().to_dict())

Train: (16011, 25) {0: 15046, 1: 965}
Valid: (3431, 25) {0: 3287, 1: 144}
Test : (3431, 25) {0: 3248, 1: 183}


In [7]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd
import numpy as np


def evaluate_probabilities(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "pr_auc": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        )
    }


weight_settings = [
    {"name": "weight_3", "scale_pos_weight": 3},
    {"name": "weight_5", "scale_pos_weight": 5},
    {"name": "weight_10", "scale_pos_weight": 10},
    {"name": "weight_20", "scale_pos_weight": 20},
    {"name": "sqrt_balanced", "auto_class_weights": "SqrtBalanced"},
    {"name": "balanced", "auto_class_weights": "Balanced"}
]

trained_weight_models = {}
weight_results = []

for setting in weight_settings:
    model_name = setting["name"]

    print("=" * 70)
    print("학습:", model_name)

    model_params = {
        "iterations": 1200,
        "learning_rate": 0.03,
        "depth": 6,
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": 42,
        "allow_writing_files": False,
        "verbose": 0
    }

    if "scale_pos_weight" in setting:
        model_params["scale_pos_weight"] = setting["scale_pos_weight"]

    if "auto_class_weights" in setting:
        model_params["auto_class_weights"] = setting["auto_class_weights"]

    model = CatBoostClassifier(**model_params)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=120,
        verbose=False
    )

    valid_prob = model.predict_proba(X_valid)[:, 1]

    metrics = evaluate_probabilities(
        y_valid,
        valid_prob,
        threshold=0.5
    )

    trained_weight_models[model_name] = model

    weight_results.append({
        "setting": model_name,
        "best_iteration": model.get_best_iteration(),
        **metrics
    })

    print({
        key: round(value, 4)
        if isinstance(value, float)
        else value
        for key, value in metrics.items()
    })

weight_result_df = (
    pd.DataFrame(weight_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(weight_result_df)

print(
    "검증셋 무작위 PR-AUC 기준:",
    round(y_valid.mean(), 4)
)

학습: weight_3
{'pr_auc': 0.0782, 'roc_auc': 0.6145, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
학습: weight_5
{'pr_auc': 0.0786, 'roc_auc': 0.6118, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
학습: weight_10
{'pr_auc': 0.0772, 'roc_auc': 0.6156, 'precision': 0.1905, 'recall': 0.0556, 'f1': 0.086}
학습: weight_20
{'pr_auc': 0.0863, 'roc_auc': 0.6204, 'precision': 0.0561, 'recall': 0.6389, 'f1': 0.1032}
학습: sqrt_balanced
{'pr_auc': 0.0749, 'roc_auc': 0.6197, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
학습: balanced
{'pr_auc': 0.0775, 'roc_auc': 0.6078, 'precision': 0.0646, 'recall': 0.25, 'f1': 0.1027}


,setting,best_iteration,pr_auc,roc_auc,precision,recall,f1
0,weight_20,88,0.086259,0.620382,0.056132,0.638889,0.103197
1,weight_5,236,0.078612,0.611770,0.000000,0.000000,0.000000
2,weight_3,192,0.078162,0.614496,0.000000,0.000000,0.000000
3,balanced,55,0.077526,0.607761,0.064632,0.250000,0.102710
4,weight_10,46,0.077189,0.615628,0.190476,0.055556,0.086022
5,sqrt_balanced,141,0.074908,0.619705,0.000000,0.000000,0.000000


검증셋 무작위 PR-AUC 기준: 0.042


In [8]:
threshold_comparison = []

for model_name, model in trained_weight_models.items():
    valid_prob = model.predict_proba(X_valid)[:, 1]

    for threshold in np.arange(0.05, 0.96, 0.01):
        pred = (valid_prob >= threshold).astype(int)

        threshold_comparison.append({
            "setting": model_name,
            "threshold": round(float(threshold), 2),
            "precision": precision_score(
                y_valid,
                pred,
                zero_division=0
            ),
            "recall": recall_score(
                y_valid,
                pred,
                zero_division=0
            ),
            "f1": f1_score(
                y_valid,
                pred,
                zero_division=0
            )
        })

threshold_comparison_df = pd.DataFrame(
    threshold_comparison
)

best_threshold_by_model = (
    threshold_comparison_df
    .sort_values(
        ["setting", "f1"],
        ascending=[True, False]
    )
    .groupby("setting", as_index=False)
    .first()
    .sort_values("f1", ascending=False)
)

display(best_threshold_by_model)

,setting,threshold,precision,recall,f1
1,sqrt_balanced,0.23,0.114754,0.145833,0.128440
4,weight_3,0.19,0.140351,0.111111,0.124031
2,weight_10,0.48,0.121622,0.125000,0.123288
5,weight_5,0.24,0.091228,0.180556,0.121212
3,weight_20,0.57,0.102564,0.138889,0.117994
0,balanced,0.55,0.134615,0.097222,0.112903


In [9]:
model_summary = weight_result_df.merge(
    best_threshold_by_model[
        [
            "setting",
            "threshold",
            "precision",
            "recall",
            "f1"
        ]
    ],
    on="setting",
    how="left",
    suffixes=("_at_05", "_best")
)

display(
    model_summary.sort_values(
        ["pr_auc", "f1_best"],
        ascending=False
    )
)

best_setting = (
    model_summary
    .sort_values(
        ["pr_auc", "f1_best"],
        ascending=False
    )
    .iloc[0]["setting"]
)

best_threshold = float(
    best_threshold_by_model.loc[
        best_threshold_by_model["setting"] == best_setting,
        "threshold"
    ].iloc[0]
)

best_model = trained_weight_models[best_setting]

print("선택 모델:", best_setting)
print("선택 임계값:", best_threshold)

,setting,best_iteration,pr_auc,roc_auc,precision_at_05,recall_at_05,f1_at_05,threshold,precision_best,recall_best,f1_best
0,weight_20,88,0.086259,0.620382,0.056132,0.638889,0.103197,0.57,0.102564,0.138889,0.117994
1,weight_5,236,0.078612,0.611770,0.000000,0.000000,0.000000,0.24,0.091228,0.180556,0.121212
2,weight_3,192,0.078162,0.614496,0.000000,0.000000,0.000000,0.19,0.140351,0.111111,0.124031
3,balanced,55,0.077526,0.607761,0.064632,0.250000,0.102710,0.55,0.134615,0.097222,0.112903
4,weight_10,46,0.077189,0.615628,0.190476,0.055556,0.086022,0.48,0.121622,0.125000,0.123288
5,sqrt_balanced,141,0.074908,0.619705,0.000000,0.000000,0.000000,0.23,0.114754,0.145833,0.128440


선택 모델: weight_20
선택 임계값: 0.57


In [10]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# PR-AUC가 가장 높은 모델 선택
final_setting = "weight_20"
final_threshold = 0.57

selected_model = trained_weight_models[final_setting]

test_prob = selected_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= final_threshold).astype(int)

test_metrics = {
    "model": final_setting,
    "threshold": final_threshold,
    "pr_auc": average_precision_score(y_test, test_prob),
    "roc_auc": roc_auc_score(y_test, test_prob),
    "precision": precision_score(
        y_test, test_pred, zero_division=0
    ),
    "recall": recall_score(
        y_test, test_pred, zero_division=0
    ),
    "f1": f1_score(
        y_test, test_pred, zero_division=0
    )
}

print("최종 테스트 결과")

for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("\n분류 보고서")
print(
    classification_report(
        y_test,
        test_pred,
        digits=4,
        zero_division=0
    )
)

print("혼동행렬")
print(confusion_matrix(y_test, test_pred))

print(
    "\n테스트셋 무작위 PR-AUC 기준:",
    round(y_test.mean(), 4)
)

최종 테스트 결과
model: weight_20
threshold: 0.5700
pr_auc: 0.0794
roc_auc: 0.6055
precision: 0.0816
recall: 0.0874
f1: 0.0844

분류 보고서
              precision    recall  f1-score   support

           0     0.9484    0.9446    0.9465      3248
           1     0.0816    0.0874    0.0844       183

    accuracy                         0.8989      3431
   macro avg     0.5150    0.5160    0.5155      3431
weighted avg     0.9021    0.8989    0.9005      3431

혼동행렬
[[3068  180]
 [ 167   16]]

테스트셋 무작위 PR-AUC 기준: 0.0533


In [11]:
from catboost import CatBoostClassifier
import pandas as pd

X_train_final = pd.concat(
    [X_train, X_valid],
    ignore_index=True
)

y_train_final = pd.concat(
    [y_train, y_valid],
    ignore_index=True
)

final_model = CatBoostClassifier(
    iterations=selected_model.get_best_iteration() + 1,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    scale_pos_weight=20,
    random_seed=42,
    allow_writing_files=False,
    verbose=0
)

final_model.fit(
    X_train_final,
    y_train_final,
    cat_features=cat_feature_indices,
    verbose=False
)

print("최종 모델 학습 완료")
print("사용 트리 수:", final_model.tree_count_)

최종 모델 학습 완료
사용 트리 수: 89


In [12]:
reference_prob = final_model.predict_proba(
    X_train_final
)[:, 1]

risk_reference = pd.DataFrame({
    "risk_score": reference_prob
}).sort_values("risk_score").reset_index(drop=True)

risk_reference.to_csv(
    "models/business_risk_reference.csv",
    index=False,
    encoding="utf-8-sig"
)

print(risk_reference.describe())

         risk_score
count  19442.000000
mean       0.473219
std        0.161957
min        0.012876
25%        0.341684
50%        0.514782
75%        0.602227
max        0.866865


In [13]:
from pathlib import Path
import json

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "business_model.cbm"
metadata_path = MODEL_DIR / "business_model_metadata.json"

final_model.save_model(str(model_path))

metadata = {
    "model_name": "CatBoostClassifier",
    "model_version": "business-v2",
    "prediction_target": "개업 후 2년 내 폐업 여부",
    "service_usage": "상대적 영업지속 위험 백분위 산출",
    "scale_pos_weight": 20,
    "classification_threshold": final_threshold,
    "feature_cols": feature_cols,
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "validation_metrics": {
        "pr_auc": 0.086259,
        "roc_auc": 0.620382
    },
    "test_metrics": {
        key: (
            float(value)
            if isinstance(value, (float, int))
            and key not in ["model"]
            else value
        )
        for key, value in test_metrics.items()
    },
    "disclaimer": (
        "공개 인허가 및 개업 당시 지역 특성에 기반한 "
        "상대위험 참고모델이며 개별 업체의 실제 폐업을 "
        "확정하지 않습니다."
    )
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2
    )

print("모델 저장:", model_path)
print("메타데이터 저장:", metadata_path)
print("기준점수 저장:", MODEL_DIR / "business_risk_reference.csv")

모델 저장: models\business_model.cbm
메타데이터 저장: models\business_model_metadata.json
기준점수 저장: models\business_risk_reference.csv
